<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


# Subscribing to the ToF sensor data

The previous notebooks inspected a ToF reading and followed its path from the sensor to a ROS 2 topic. Using these measurements in a ROS 2 program requires receiving each message, retaining the relevant fields, and deciding how to present the measurements.

This notebook completes a subscriber for the downward sensor and uses it to observe changes in distance. Its callback stores arriving readings, while a timer prints the latest stored range once per second. Following these two operations explains both how the display responds to movement and what a repeated number means.

This notebook addresses the following questions:

- How does a subscriber receive and store a ToF reading?

- How can the input rate and display rate be observed separately?

- What information is retained from each message?

- How should reported values and continued message arrival be checked?

## Getting to a terminal that can see the Duckiedrone

Use the shell inside `ros2-tof-bottom` opened in [Time-of-Flight (ToF) Sensor Fundamentals](./1-tof-basics.ipynb#gathering-tof-measurements-from-a-duckiedrone). If that session has closed, repeat its physical or virtual connection steps, followed by `docker exec` to enter the container. For a virtual Duckiedrone, keep the Duckiematrix running with the Duckiedrone attached to an entity or no sensor measurements will be produced.

The ROS 2 inspection commands run in this bridge shell. Edit the source in the LX editor, then run `dts code build` and `dts code workbench` from the LX project folder in a separate base-station terminal.

Replace `DUCKIEDRONE_NAME` in commands with the configured name.

## Look at the data first

Before connecting the program, inspect the endpoint it will use:

```bash
ros2 topic info -v /DUCKIEDRONE_NAME/bottom_tof_driver_node/range
```

The output identifies the message type, discovered publishers and subscribers, and their quality-of-service (QoS) settings. Look for `sensor_msgs/msg/Range` and a publisher belonging to the ToF bridge. The subscriber list depends on the services and inspection tools currently running.

If the topic is missing, list the available names:

```bash
ros2 topic list
```

Use the exact downward-ToF topic shown there. An absent topic or publisher calls for checking the driver and bridge before proceeding to the subscriber code.

Next, observe message arrivals:

```bash
ros2 topic hz /DUCKIEDRONE_NAME/bottom_tof_driver_node/range
```

Let several reporting windows appear, record the observed rate, and stop the command with `Ctrl+C`. The default physical bottom driver requests 30 Hz, so the reported `average rate` of messages received during this observation should be close to 30:

```text
root@amelia:/code/src/dt-ros2-interface# ros2 topic hz /amelia/bottom_tof_driver_node/range
average rate: 29.859
	min: 0.026s max: 0.041s std dev: 0.00312s window: 31
average rate: 29.843
	min: 0.026s max: 0.041s std dev: 0.00312s window: 61
average rate: 29.863
	min: 0.025s max: 0.043s std dev: 0.00347s window: 91
average rate: 29.863
	min: 0.025s max: 0.043s std dev: 0.00334s window: 121
```

Finally, inspect one message:

```bash
ros2 topic echo --once /DUCKIEDRONE_NAME/bottom_tof_driver_node/range
```

The command prints one received message and exits. If it keeps waiting, use `Ctrl+C` to stop it and investigate the stream.

Identify `range`, `header.stamp`, and `header.frame_id`. These are the fields the subscriber will retain. Also note `min_range` and `max_range`, which will help interpret the displayed values.


## Activity: observe changes in the downward range

With the downward stream established, build a subscriber and move a target to observe how the reported range responds. Comparing incoming messages with the once-per-second display shows how a program can retain and reuse a sensor reading.

### 1. Write the subscriber

<a id="activity-1"></a>

Open [tof_subscriber.py](../packages/solution/solution/tof_subscriber.py) in the LX editor. In the learner starter, complete the two `TO IMPLEMENT` blocks. In the solution version, these blocks already contain the implementation; inspect how they perform the following operations.

The surrounding code reads the node parameters, constructs the topic names, creates `self.latest_readings`, and starts a timer.



#### In `__init__`, create the subscription

For each `full_topic`, create a subscription using:

- `Range` as the message type;

- `full_topic` as the topic name;

- `self._make_callback(name)` as the callback;

- `qos_profile_sensor_data` as the QoS profile.

Append the returned subscription to `self.subs`.

Calling `_make_callback(name)` creates a callback associated with that sensor name. ROS 2 subsequently calls it whenever a message is delivered.

The sensor-data QoS profile requests best-effort delivery with volatile durability. The current bridge source creates a publisher with the default reliable and volatile policies and a queue depth of one. These reliability and durability settings are compatible: a best-effort subscriber can receive from a reliable publisher.

Compare this explanation with the publisher settings shown by `ros2 topic info -v` on the running Duckiedrone. Compatibility between the endpoints allows delivery; their profiles can differ.

<a id="unpack-one-message"></a>

#### In `_make_callback`, unpack one message

Create a `ToFReading` and store it in `self.latest_readings[name]`, using:

- `name` for `sensor_name`;

- `float(msg.range)` for `range_m`;

- `msg.header.stamp.sec + msg.header.stamp.nanosec * 1e-9` for `stamp_sec`;

- `msg.header.frame_id` for `frame_id`.

The timestamp expression converts the message's seconds and nanoseconds into seconds. The callback preserves the source timestamp supplied through the bridge.

The sensor name comes from the subscription configuration, so the dictionary remains usable when a message has an empty frame identifier.

Each arriving message replaces the previous entry for that sensor. The timer calls `cb_log` once per second and prints the range currently stored in the dictionary. Several messages can therefore update the entry between printed lines.

This implementation retains the range, timestamp, frame identifier, and sensor name. Its `ToFReading` structure would need additional fields to retain the message's range limits.


### 2. Run the node on the Duckiedrone

<a id="activity-2"></a>

Save the source edits. From the LX project root in the base-station terminal, build the code for the Duckiedrone:

```bash
dts code build -R DUCKIEDRONE_NAME
```

After the build succeeds, start the workbench:

```bash
dts code workbench -R DUCKIEDRONE_NAME
```

The workbench terminal displays the listener's output. The following illustrative lines use the name `amelia` and a reported range below the advertised minimum:

```text
[tof_listener_node-1] [INFO] [tof_listener_node]: Subscribing to /amelia/bottom_tof_driver_node/range  (sensor: bottom)
[tof_listener_node-1] [INFO] [tof_listener_node]: ranges=[bottom=0.018m]
```

The startup line reports the topic constructed from the launch configuration. Check that it matches the topic inspected earlier. The code prints this line before creating the subscription; receiving a reading is the next observable step.

Keep the workbench running for [step 3](#activity-3). Use `Ctrl+C` in this terminal to stop the run. After further source edits, save, rebuild, and restart the workbench.

#### When nothing arrives

Before the first callback stores a reading, the timer prints:

```text
Waiting for ToF data...
```

If this continues:

1. Check that both implementation blocks are complete and the saved code was rebuilt.

2. Compare the startup topic with the topic listed in the bridge shell.

3. Inspect that topic's publisher and QoS settings, then check for arriving messages with `ros2 topic hz`.

4. In the Duckiedrone device shell, check that `driver-tof-bottom` and `ros2-tof-bottom` are running. For a virtual Duckiedrone, also check the running Duckiematrix and entity attachment.

Use these observations to locate the interruption. For example, messages visible in the bridge shell narrow the investigation to the listener's configuration, subscription, and communication environment.

#### What a repeated line means

After the first message arrives, the dictionary retains a reading until the next callback replaces it. The timer continues printing that stored range during any interruption in input.

A repeated value can therefore arise from a stationary scene, rounding to the displayed millimeters, or reuse of a cached reading. Inspect message arrivals and timestamps when freshness matters.

For comparison with the once-per-second display, rerun `ros2 topic hz` in the bridge shell while the listener is running. The two terminals observe different operations: incoming messages and timer-driven summaries.


### 3. Make the number move

<a id="activity-3"></a>

With the listener running, vary the distance between the downward sensor and a broad, flat target.

Keep a physical Duckiedrone disarmed, with its motors stopped. Support it level above the target, initially leaving approximately 20 to 30 cm between the sensor and the surface. Move the target slowly toward and away from the sensor while keeping it inside the advertised range interval.

Observe the direction of the change and the updates in the once-per-second display. A quick movement can occur between printed lines, because the callback replaces intermediate readings before the timer prints again.

For a virtual Duckiedrone, observe the downward range as the attached entity's height above the simulated surface changes. Interpret the values using that scene and the limits in its messages.

Compare the displayed readings with three cases:

- __Inside the advertised interval.__ A value such as `0.200m` lies between the default bounds of 0.04 m and 3.6 m. Its usefulness for a particular task also depends on the target geometry, measurement quality, and freshness.

- __Below the advertised minimum.__ The physical floor recording in [Time-of-Flight (ToF) Sensor Fundamentals](./1-tof-basics.ipynb) contains values near `0.018m`. The current subscriber stores and prints such values unchanged.

- __An unavailable range represented by the bridge.__ A weak or rejected return can produce `999.000m`. This is the bridge's marker for an unusable reading.

To explore the last case on a physical Duckiedrone, direct the downward sensor toward open space while keeping the motors stopped. Observe the actual response: the scene and sensor conditions determine whether usable readings, marker values, or gaps occur.

The listener formats all stored values in the same way. A consumer that uses them for a decision should check for finite values within the message's range bounds and apply a freshness requirement appropriate to its task.

## Going further

The following extensions build on the existing subscriber:

- __Retain and check the limits.__ Add `min_range` and `max_range` to `ToFReading` and copy them in the callback. Have the logger distinguish finite values within those bounds from readings to discard.

- __Expose the stored timestamp.__ Print `stamp_sec` alongside the range. Observe whether it advances when consecutive displayed distances are equal.

- __Track arrival age.__ Record the local monotonic time whenever the callback receives a message. The logger can then report elapsed time since the last arrival. Define an acceptable age according to the intended application.

- __Inspect the stream directly.__ Run the following command in the bridge shell, then stop it with `Ctrl+C`:

```bash
ros2 topic echo /DUCKIEDRONE_NAME/bottom_tof_driver_node/range --field range
```

The command prints the range field of messages it receives, allowing comparison with the timer's summaries. Full-message inspection remains useful when timestamps and range limits are needed.

## Summary and next steps

The subscriber connects to the downward-ToF topic, converts selected message fields into a `ToFReading`, and stores the latest received value. Its timer displays that stored range once per second.

The movement activity connects changes in the scene to changes in the reported distance. Interpreting the display also requires the message's range limits and evidence that readings remain recent.

[All Five Sensors](./5-all-five-sensors.ipynb) extends the same subscriber to the other four ToF sensors.

## Further reading

- [ROS 2 quality-of-service settings](https://docs.ros.org/en/jazzy/Concepts/Intermediate/About-Quality-of-Service-Settings.html): delivery policies and endpoint compatibility.

- [Range message definition](https://github.com/ros2/common_interfaces/blob/jazzy/sensor_msgs/msg/Range.msg): message fields and range bounds.

- [Duckietown ToF bridge](https://github.com/duckietown/dt-ros2-interface/blob/ente/packages/tof_driver/tof_driver/tof_driver_node.py): publisher configuration and conversion of incoming readings.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()